<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/14_catalyst_optimizer_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("catalyst") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

data = [("Raj",   "Engineering", 50000),
        ("Amit",  "Engineering", 60000),
        ("John",  "Engineering", 90000),
        ("Priya", "Marketing",   70000),
        ("Karan", "Marketing",   45000),
        ("Sara",  "HR",          80000),
        ("Neha",  "HR",          80000),   # same salary as Sara
        ("Raj2",  "HR",          55000)]

columns = ["name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|  Raj|Engineering| 50000|
| Amit|Engineering| 60000|
| John|Engineering| 90000|
|Priya|  Marketing| 70000|
|Karan|  Marketing| 45000|
| Sara|         HR| 80000|
| Neha|         HR| 80000|
| Raj2|         HR| 55000|
+-----+-----------+------+



In [3]:
df.select("name", "salary").filter(F.col("salary")>50000).explain(True) #true shows all 4 stages

== Parsed Logical Plan ==
'Filter '`>`('salary, 50000)
+- Project [name#0, salary#2L]
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Analyzed Logical Plan ==
name: string, salary: bigint
Filter (salary#2L > cast(50000 as bigint))
+- Project [name#0, salary#2L]
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Optimized Logical Plan ==
Project [name#0, salary#2L]
+- Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Physical Plan ==
*(1) Project [name#0, salary#2L]
+- *(1) Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- *(1) Scan ExistingRDD[name#0,department#1,salary#2L]



In [4]:
df.select("name", "salary").filter(F.col("salary")>50000).explain() #without true it just shows physical plan

== Physical Plan ==
*(1) Project [name#0, salary#2L]
+- *(1) Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- *(1) Scan ExistingRDD[name#0,department#1,salary#2L]




In [5]:
df.filter(F.col("salary")>50000).select("name", "salary").explain(True)

== Parsed Logical Plan ==
'Project ['name, 'salary]
+- Filter (salary#2L > cast(50000 as bigint))
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Analyzed Logical Plan ==
name: string, salary: bigint
Project [name#0, salary#2L]
+- Filter (salary#2L > cast(50000 as bigint))
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Optimized Logical Plan ==
Project [name#0, salary#2L]
+- Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- LogicalRDD [name#0, department#1, salary#2L], false

== Physical Plan ==
*(1) Project [name#0, salary#2L]
+- *(1) Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- *(1) Scan ExistingRDD[name#0,department#1,salary#2L]



In [6]:
df.filter(F.col("salary")>50000).select("name", "salary").explain()

== Physical Plan ==
*(1) Project [name#0, salary#2L]
+- *(1) Filter (isnotnull(salary#2L) AND (salary#2L > 50000))
   +- *(1) Scan ExistingRDD[name#0,department#1,salary#2L]




In [7]:
#the order doesnt matter - both query's are identical and gets executes in same time